In [0]:
# ============================================================
# 06_gold_dimensions
# ============================================================
# Purpose  : Load all 4 dimension tables into Gold
#            dim_date, dim_location, dim_violation, dim_restaurant (SCD2)
# Reads    : silver.inspections_unified, silver.violations_unified
# Writes   : gold.dim_date, gold.dim_location,
#            gold.dim_violation, gold.dim_restaurant
# Run order: Must run before 07_gold_facts
# ============================================================

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import IntegerType, DoubleType, BooleanType, DateType
from datetime import date

CATALOG = "final_project_databi_group8"

UNIFIED_INSP  = f"{CATALOG}.silver.inspections_unified"
UNIFIED_VIOLS = f"{CATALOG}.silver.violations_unified"

DIM_DATE       = f"{CATALOG}.gold.dim_date"
DIM_LOCATION   = f"{CATALOG}.gold.dim_location"
DIM_VIOLATION  = f"{CATALOG}.gold.dim_violation"
DIM_RESTAURANT = f"{CATALOG}.gold.dim_restaurant"

# Create schemas if they don't exist
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`bronze`")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`silver`")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`gold`")
print(f" Schemas ready: bronze, silver, gold")

print(f"Catalog        : {CATALOG}")
print(f"Source insp    : {UNIFIED_INSP}")
print(f"Source viols   : {UNIFIED_VIOLS}")
print()
print("Target tables:")
for t in [DIM_DATE, DIM_LOCATION, DIM_VIOLATION, DIM_RESTAURANT]:
    print(f"  {t}")

In [0]:
for tbl in [UNIFIED_INSP, UNIFIED_VIOLS]:
    try:
        spark.table(tbl).limit(1).collect()
        print(f" Found: {tbl}")
    except Exception as e:
        raise Exception(f"Table not found: {tbl}\nRun Silver notebooks first.\nError: {e}")

In [0]:
# ── dim_date ──────────────────────────────────────────────────
# date_key = YYYYMMDD integer (e.g. 20260408)
# Covers 2010-01-01 → 2026-12-31 = 6,209 rows

dim_date_df = spark.sql("""
    SELECT explode(sequence(
        to_date('2010-01-01'),
        to_date('2026-12-31'),
        interval 1 day
    )) AS full_date
""")

dim_date_df = (
    dim_date_df
    .withColumn("date_key",
        (F.year("full_date") * 10000 +
         F.month("full_date") * 100 +
         F.dayofmonth("full_date")).cast(IntegerType()))
    .withColumn("year",               F.year("full_date").cast(IntegerType()))
    .withColumn("quarter",            F.quarter("full_date").cast(IntegerType()))
    .withColumn("quarter_name",       F.concat(F.lit("Q"), F.quarter("full_date").cast("string")))
    .withColumn("month",              F.month("full_date").cast(IntegerType()))
    .withColumn("month_name",         F.date_format("full_date", "MMMM"))
    .withColumn("month_abbr",         F.date_format("full_date", "MMM"))
    .withColumn("week_of_year",       F.weekofyear("full_date").cast(IntegerType()))
    .withColumn("day_of_month",       F.dayofmonth("full_date").cast(IntegerType()))
    .withColumn("day_name",           F.date_format("full_date", "EEEE"))
    .withColumn("day_of_week",        F.dayofweek("full_date").cast(IntegerType()))
    .withColumn("is_weekend",
        F.dayofweek("full_date").isin(1, 7).cast(BooleanType()))
    .withColumn("first_day_of_month", F.trunc("full_date", "month").cast(DateType()))
    .withColumn("last_day_of_month",  F.last_day("full_date").cast(DateType()))
    .withColumn("dw_created_ts",      F.current_timestamp())
    .select(
        "date_key", "full_date", "year", "quarter", "quarter_name",
        "month", "month_name", "month_abbr", "week_of_year",
        "day_of_month", "day_name", "day_of_week", "is_weekend",
        "first_day_of_month", "last_day_of_month", "dw_created_ts"
    )
)

(
    dim_date_df
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(DIM_DATE)
)

count = spark.table(DIM_DATE).count()
print(f" {DIM_DATE}")
print(f"  Rows: {count:,}  (expected 6,209)")
display(spark.table(DIM_DATE).limit(5))

In [0]:
# ── dim_location — Type 1 ─────────────────────────────────────
# Business key: street_address + zip_code + city_source
# Type 1: corrections overwrite, no history needed

loc_src = (
    spark.table(UNIFIED_INSP)
    .select("street_address", "zip_code", "city", "state",
            "latitude", "longitude", "city_source")
    .distinct()
    .filter(F.col("street_address").isNotNull())
)

w_loc = Window.orderBy("city_source", "street_address", "zip_code")
loc_src = loc_src.withColumn(
    "location_key",
    F.row_number().over(w_loc).cast(IntegerType())
)

loc_src = (
    loc_src
    .withColumn("dw_created_ts", F.current_timestamp())
    .withColumn("dw_updated_ts", F.current_timestamp())
    .select(
        "location_key", "street_address", "zip_code",
        "city_source", "city", "state",
        "latitude", "longitude",
        "dw_created_ts", "dw_updated_ts"
    )
)

try:
    spark.table(DIM_LOCATION).limit(1).collect()
    loc_src.createOrReplaceTempView("loc_src_vw")
    spark.sql(f"""
        MERGE INTO {DIM_LOCATION} AS target
        USING loc_src_vw AS source
        ON  target.street_address = source.street_address
        AND target.zip_code       = source.zip_code
        AND target.city_source    = source.city_source
        WHEN MATCHED AND (
            COALESCE(CAST(target.latitude  AS STRING), '') != COALESCE(CAST(source.latitude  AS STRING), '') OR
            COALESCE(CAST(target.longitude AS STRING), '') != COALESCE(CAST(source.longitude AS STRING), '') OR
            COALESCE(target.city, '') != COALESCE(source.city, '')
        ) THEN UPDATE SET
            target.latitude      = source.latitude,
            target.longitude     = source.longitude,
            target.city          = source.city,
            target.state         = source.state,
            target.dw_updated_ts = source.dw_updated_ts
        WHEN NOT MATCHED THEN INSERT *
    """)
    print(f" {DIM_LOCATION} — MERGED")
except Exception:
    (
        loc_src.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(DIM_LOCATION)
    )
    print(f" {DIM_LOCATION} — CREATED (first run)")

count = spark.table(DIM_LOCATION).count()
print(f"  Rows: {count:,}")
display(spark.table(DIM_LOCATION).limit(3))

In [0]:
# ── dim_violation — Type 1 ────────────────────────────────────
# Business key: violation_code + city_source
# Chicago codes and Dallas codes stored together
# city_source keeps them separate — Code 42 CHI != Code 42 DAL

viol_src = (
    spark.table(UNIFIED_VIOLS)
    .select("violation_code", "violation_description", "city_source")
    .filter(F.col("violation_code").isNotNull() & (F.col("violation_code") != ""))
    .distinct()
)

w_viol = Window.orderBy("city_source", "violation_code")
viol_src = viol_src.withColumn(
    "violation_key",
    F.row_number().over(w_viol).cast(IntegerType())
)

viol_src = (
    viol_src
    .withColumn("dw_created_ts", F.current_timestamp())
    .withColumn("dw_updated_ts", F.current_timestamp())
    .select(
        "violation_key", "violation_code",
        "violation_description", "city_source",
        "dw_created_ts", "dw_updated_ts"
    )
)

try:
    spark.table(DIM_VIOLATION).limit(1).collect()
    viol_src.createOrReplaceTempView("viol_src_vw")
    spark.sql(f"""
        MERGE INTO {DIM_VIOLATION} AS target
        USING viol_src_vw AS source
        ON  target.violation_code = source.violation_code
        AND target.city_source    = source.city_source
        WHEN MATCHED AND COALESCE(target.violation_description, '') != COALESCE(source.violation_description, '')
        THEN UPDATE SET
            target.violation_description = source.violation_description,
            target.dw_updated_ts         = source.dw_updated_ts
        WHEN NOT MATCHED THEN INSERT *
    """)
    print(f" {DIM_VIOLATION} — MERGED")
except Exception:
    (
        viol_src.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(DIM_VIOLATION)
    )
    print(f" {DIM_VIOLATION} — CREATED (first run)")

count = spark.table(DIM_VIOLATION).count()
print(f"  Rows: {count:,}  (expected ~115)")
display(spark.table(DIM_VIOLATION).orderBy("city_source", "violation_code").limit(10))

In [0]:
# ── dim_restaurant — History-Derived SCD2 ────────────────────
# Business key : restaurant_name + street_address + city_source
# Tracked attrs: aka_name, facility_type, license_number
# SCD columns  : effective_date, expiry_date, is_current

# Step 1: All distinct restaurant attribute versions from Silver
restaurant_versions = (
    spark.table(UNIFIED_INSP)
    .select(
        "restaurant_name", "street_address", "city_source",
        "aka_name", "facility_type", "license_number",
        "city", "state", "inspection_date"
    )
    .filter(F.col("restaurant_name").isNotNull())
    .filter(F.col("street_address").isNotNull())
)

# Step 2: First inspection date per distinct attribute version
# effective_date = when this version first appeared in inspection history
first_seen = (
    restaurant_versions
    .groupBy(
        "restaurant_name", "street_address", "city_source",
        "aka_name", "facility_type", "license_number",
        "city", "state"
    )
    .agg(F.min("inspection_date").alias("effective_date"))
)

print(f"Distinct restaurant attribute versions: {first_seen.count():,}")
display(first_seen.orderBy("restaurant_name", "street_address", "effective_date").limit(5))

In [0]:
# Step 3: Assign expiry dates using window function
# expiry_date = one day before next version's effective_date
# Last version gets 9999-12-31

w_scd2 = Window.partitionBy(
    "restaurant_name", "street_address", "city_source"
).orderBy("effective_date")

dim_rest_df = (
    first_seen
    .withColumn("next_effective_date",
        F.lead("effective_date").over(w_scd2))
    .withColumn("expiry_date",
        F.when(F.col("next_effective_date").isNull(),
               F.lit("9999-12-31").cast(DateType()))
         .otherwise(F.date_sub(F.col("next_effective_date"), 1)))
    .withColumn("is_current",
        (F.col("expiry_date") == F.lit("9999-12-31").cast(DateType())).cast(BooleanType()))
    .drop("next_effective_date")
)

# Step 4: Assign integer surrogate keys
w_key = Window.orderBy("city_source", "restaurant_name", "street_address", "effective_date")
dim_rest_df = dim_rest_df.withColumn(
    "restaurant_key",
    F.row_number().over(w_key).cast(IntegerType())
)

# Step 5: Add audit column and final column selection (matches PDF model)
dim_rest_df = (
    dim_rest_df
    .withColumn("dw_created_ts", F.current_timestamp())
    .select(
        "restaurant_key",
        "restaurant_name",
        "street_address",
        "city_source",
        "aka_name",
        "facility_type",
        "license_number",
        "city",
        "state",
        "effective_date",
        "expiry_date",
        "is_current",
        "dw_created_ts"
    )
)

total_versions     = dim_rest_df.count()
current_records    = dim_rest_df.filter(F.col("is_current") == True).count()
historical_records = dim_rest_df.filter(F.col("is_current") == False).count()

print(f"Total restaurant versions        : {total_versions:,}")
print(f"Current records  (is_current=T)  : {current_records:,}")
print(f"Historical records (is_current=F): {historical_records:,}")
display(dim_rest_df.orderBy("restaurant_name", "street_address", "effective_date").limit(8))

In [0]:
# Step 6: Write dim_restaurant
# First run  → create table with full History-Derived SCD2 history
# Subsequent → MERGE: close changed rows, insert new versions

try:
    existing = spark.table(DIM_RESTAURANT)
    existing.limit(1).collect()

    # MERGE SCD2 (subsequent runs)
    current_silver = (
        spark.table(UNIFIED_INSP)
        .select(
            "restaurant_name", "street_address", "city_source",
            "aka_name", "facility_type", "license_number",
            "city", "state"
        )
        .filter(F.col("restaurant_name").isNotNull())
        .filter(F.col("street_address").isNotNull())
        .distinct()
    )

    current_silver.createOrReplaceTempView("current_silver_vw")

    # Step A: Close records where tracked attributes changed
    spark.sql(f"""
        MERGE INTO {DIM_RESTAURANT} AS target
        USING current_silver_vw AS source
        ON  target.restaurant_name = source.restaurant_name
        AND target.street_address  = source.street_address
        AND target.city_source     = source.city_source
        AND target.is_current      = true
        WHEN MATCHED AND (
            COALESCE(target.aka_name,       '') != COALESCE(source.aka_name,       '') OR
            COALESCE(target.facility_type,  '') != COALESCE(source.facility_type,  '') OR
            COALESCE(target.license_number, '') != COALESCE(source.license_number, '')
        ) THEN UPDATE SET
            target.is_current  = false,
            target.expiry_date = current_date()
    """)

    # Step B: Insert new versions for changed + brand new restaurants
    max_key = spark.table(DIM_RESTAURANT).agg(F.max("restaurant_key")).first()[0] or 0

    changed_or_new = (
        current_silver.alias("s")
        .join(
            spark.table(DIM_RESTAURANT)
            .filter(F.col("is_current") == True)
            .select("restaurant_name", "street_address", "city_source")
            .alias("t"),
            on=["restaurant_name", "street_address", "city_source"],
            how="left_anti"
        )
    )

    insert_count = changed_or_new.count()
    if insert_count > 0:
        w_new = Window.orderBy("city_source", "restaurant_name", "street_address")
        new_rows = (
            changed_or_new
            .withColumn("restaurant_key",
                (F.row_number().over(w_new) + max_key).cast(IntegerType()))
            .withColumn("effective_date", F.current_date().cast(DateType()))
            .withColumn("expiry_date",    F.lit("9999-12-31").cast(DateType()))
            .withColumn("is_current",     F.lit(True).cast(BooleanType()))
            .withColumn("dw_created_ts",  F.current_timestamp())
            .select(
                "restaurant_key", "restaurant_name", "street_address", "city_source",
                "aka_name", "facility_type", "license_number", "city", "state",
                "effective_date", "expiry_date", "is_current", "dw_created_ts"
            )
        )
        new_rows.write.format("delta").mode("append").saveAsTable(DIM_RESTAURANT)
        print(f" {DIM_RESTAURANT} — MERGED: {insert_count:,} new rows inserted")
    else:
        print(f" {DIM_RESTAURANT} — No changes detected")

except Exception:
    # FIRST RUN — full History-Derived SCD2
    (
        dim_rest_df
        .write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(DIM_RESTAURANT)
    )
    print(f" {DIM_RESTAURANT} — CREATED (first run, full SCD2 history)")

total   = spark.table(DIM_RESTAURANT).count()
current = spark.table(DIM_RESTAURANT).filter(F.col("is_current") == True).count()
print(f"  Total rows     : {total:,}")
print(f"  Current rows   : {current:,}")
print(f"  Historical rows: {total - current:,}")
display(spark.table(DIM_RESTAURANT).filter(F.col("is_current") == True).limit(5))

In [0]:
print("=== DIMENSION SUMMARY ===")
for dim, expected in [
    (DIM_DATE,       "6,209"),
    (DIM_LOCATION,   "~41,000"),
    (DIM_VIOLATION,  "~115"),
    (DIM_RESTAURANT, "~70,000+"),
]:
    try:
        count = spark.table(dim).count()
        label = dim.split(".")[-1]
        print(f"  {label:<25} : {count:>10,}  (expected {expected})")
    except Exception as e:
        print(f"  {dim} : ERROR — {e}")

print()
print("=== SCD2 version counts ===")
display(spark.sql(f"""
    SELECT is_current, COUNT(*) AS row_count
    FROM {DIM_RESTAURANT}
    GROUP BY is_current
"""))

print()
print("=== Violations by city ===")
display(spark.sql(f"""
    SELECT city_source,
           COUNT(*) AS violation_codes,
           MIN(violation_code) AS min_code,
           MAX(violation_code) AS max_code
    FROM {DIM_VIOLATION}
    GROUP BY city_source
"""))